# E2E Net Forward Pass Demo

This notebook instantiates `E2ENet`, loads a checkpoint if available, and runs a forward pass on a real Div2K validation image. 
It then displays the input image and the rendered gaussian output side by side for a quick visual check of the reconstruction.

In [1]:
import os
import sys
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)
print('Added to sys.path:', repo_root)


Added to sys.path: /work/10968/nelsoner/ls6/2dgs/Instant-GI


In [ ]:
import torch
from generalizable_model.e2e_net import E2ENet, render

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = E2ENet().to(device)
model.eval()


In [ ]:
checkpoint_path = os.path.join(repo_root, 'checkpoints', 'train_e2e_apricot-sea-9', 'epoch_last.pth')
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    print('Loaded checkpoint:', checkpoint_path)
else:
    print('Checkpoint not found:', checkpoint_path)

inference_kwargs = dict(keep_ratio=0.1)


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from generalizable_model.datasets import ImageAndPF

valid_dir = os.path.join(repo_root, 'data', 'div2k_gs', 'valid')
dataset = ImageAndPF(valid_dir)
image, _ = dataset[0]
input_tensor = image.unsqueeze(0).to(device)

with torch.no_grad():
    sampling_field, render_img, _ = model(input_tensor, **inference_kwargs)

orig_np = image.permute(1, 2, 0).clamp(0, 1).numpy()
render_np = render_img[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy()
sampling_np = sampling_field[0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_np)
axes[0].set_title('Ground truth')
axes[0].axis('off')
axes[1].imshow(render_np)
axes[1].set_title('Rendered Gaussian output')
axes[1].axis('off')
axes[2].imshow(sampling_np, cmap='gray')
axes[2].set_title('Sampling field')
axes[2].axis('off')
plt.tight_layout()
plt.show()

sampling_rgb = np.repeat(sampling_np[..., None], 3, axis=2)
combined = np.concatenate([orig_np, render_np, sampling_rgb], axis=1)
combined_bgr = cv2.cvtColor((combined * 255).astype('uint8'), cv2.COLOR_RGB2BGR)
cv2.imwrite('div2k_render_vs_gt_sampling.png', combined_bgr)
print('Saved div2k_render_vs_gt_sampling.png')


## Toy Gaussian sanity check
The cell below renders three manual Gaussians so you can verify the rasterizer produces soft colored blobs independent of the trained network.


In [ ]:
import torch
from matplotlib import pyplot as plt

toy_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
H = W = 128
xy_pixels = torch.tensor([[32.0, 40.0], [96.0, 64.0], [64.0, 96.0]], device=toy_device)
scaling_pixels = torch.tensor([[12.0, 12.0], [8.0, 24.0], [25.0, 10.0]], device=toy_device)
rotation = torch.zeros(xy_pixels.shape[0], 1, device=toy_device)
colors = torch.tensor([[1.0, 0.2, 0.2], [0.2, 1.0, 0.2], [0.2, 0.3, 1.0]], device=toy_device)
opacity = torch.ones(xy_pixels.shape[0], 1, device=toy_device) * 0.85
render_out = render(xy_pixels, scaling_pixels, rotation, colors, opacity, H, W)['render']
toy_np = render_out[0].detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(toy_np)
plt.title('Toy Gaussians (3 blobs)')
plt.axis('off')
plt.show()
